In [88]:
# ============================================================
# TASK 2.1 — Supervised ECG heartbeat classification
# ============================================================

import os, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

os.makedirs("outputs", exist_ok=True)
print("Импорты готовы. Папка outputs/ создана.")

Импорты готовы. Папка outputs/ создана.


In [89]:
# Покажем все csv/npz/npy в рабочей папке (глубина ≤ 3), чтобы найти train для 2.1
for root, dirs, files in os.walk("."):
    if root.count(os.sep) > 3:
        continue
    for f in files:
        if f.endswith((".csv", ".npz", ".npy")):
            print(os.path.join(root, f))

.\test_features.csv
.\test_features.npz
.\train.csv
.\train.npz
.\outputs\task2_1_predictions.csv


In [90]:
# Укажи пути к своим файлам:
TRAIN_PATH = "train.csv"                 # обучающая выборка с метками
TEST_PATH  = "test_features.csv"     # тестовые признаки (heartbeat_id + t_*)

assert os.path.exists(TRAIN_PATH), f"Не найден {TRAIN_PATH}"
assert os.path.exists(TEST_PATH),  f"Не найден {TEST_PATH}"

df_train = pd.read_csv(TRAIN_PATH)
df_test  = pd.read_csv(TEST_PATH)

print("train shape:", df_train.shape)
print("test  shape:", df_test.shape)
print("\nКолонки train (все):", list(df_train.columns))
print("\nКолонки test (первые 10):", list(df_test.columns[:10]))
print("Колонки test (последние 3):", list(df_test.columns[-3:]))

train shape: (4000, 142)
test  shape: (1000, 141)

Колонки train (все): ['heartbeat_id', 'prediction', 't_000', 't_001', 't_002', 't_003', 't_004', 't_005', 't_006', 't_007', 't_008', 't_009', 't_010', 't_011', 't_012', 't_013', 't_014', 't_015', 't_016', 't_017', 't_018', 't_019', 't_020', 't_021', 't_022', 't_023', 't_024', 't_025', 't_026', 't_027', 't_028', 't_029', 't_030', 't_031', 't_032', 't_033', 't_034', 't_035', 't_036', 't_037', 't_038', 't_039', 't_040', 't_041', 't_042', 't_043', 't_044', 't_045', 't_046', 't_047', 't_048', 't_049', 't_050', 't_051', 't_052', 't_053', 't_054', 't_055', 't_056', 't_057', 't_058', 't_059', 't_060', 't_061', 't_062', 't_063', 't_064', 't_065', 't_066', 't_067', 't_068', 't_069', 't_070', 't_071', 't_072', 't_073', 't_074', 't_075', 't_076', 't_077', 't_078', 't_079', 't_080', 't_081', 't_082', 't_083', 't_084', 't_085', 't_086', 't_087', 't_088', 't_089', 't_090', 't_091', 't_092', 't_093', 't_094', 't_095', 't_096', 't_097', 't_098', 't_099

In [91]:
# --- ищем колонку с меткой ---
label_candidates = ["label", "prediction", "target", "y", "class"]
label_col = next((c for c in label_candidates if c in df_train.columns), None)

if label_col is None:
    raise ValueError(
        f"В train не найдена колонка с метками. "
        f"Кандидаты: {label_candidates}. "
        f"Есть колонки: {list(df_train.columns)}"
    )

print("Колонка метки:", label_col)
print("Распределение классов:")
print(df_train[label_col].value_counts().sort_index())

# --- определяем признаки t_* ---
feature_cols = [c for c in df_test.columns if c.startswith("t_")]
print(f"\nНайдено {len(feature_cols)} признаковых колонок t_*")
print("Первые:", feature_cols[:5], "... последние:", feature_cols[-5:])

# --- проверяем, что все признаки есть в train ---
missing = [c for c in feature_cols if c not in df_train.columns]
print(f"Отсутствуют в train: {len(missing)}")
if missing:
    print("  ->", missing[:10], "...")

Колонка метки: prediction
Распределение классов:
prediction
0    2335
1    1414
2      77
3     155
4      19
Name: count, dtype: int64

Найдено 140 признаковых колонок t_*
Первые: ['t_000', 't_001', 't_002', 't_003', 't_004'] ... последние: ['t_135', 't_136', 't_137', 't_138', 't_139']
Отсутствуют в train: 0


In [92]:
# Проверка соответствия колонок
common_cols = [c for c in feature_cols if c in df_train.columns]
print(f"Общих признаковых колонок: {len(common_cols)}")

X_train = df_train[common_cols].values.astype(np.float32)
y_train = df_train[label_col].values.astype(int)

X_test     = df_test[common_cols].values.astype(np.float32)
test_ids   = df_test["heartbeat_id"].values

print("X_train:", X_train.shape, " y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("Уникальные классы в y_train:", np.unique(y_train))

# На всякий случай — сколько NaN
print("NaN в X_train:", np.isnan(X_train).sum(),
      "| NaN в X_test:", np.isnan(X_test).sum())

Общих признаковых колонок: 140
X_train: (4000, 140)  y_train: (4000,)
X_test : (1000, 140)
Уникальные классы в y_train: [0 1 2 3 4]
NaN в X_train: 0 | NaN в X_test: 0


In [93]:
pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),   # на случай NaN
    ("scaler",  StandardScaler()),
    ("clf",     RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=1,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )),
])

# Кросс-валидация, если в каждом классе достаточно образцов
class_counts = pd.Series(y_train).value_counts()
min_count = int(class_counts.min())
n_splits = int(min(5, min_count))
print(f"Мин. кол-во образцов в классе: {min_count} -> n_splits={n_splits}")

if n_splits >= 2:
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=-1)
    print(f"CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
else:
    print("⚠️ Слишком мало данных для CV — пропускаем.")

# Финальное обучение на всей обучающей выборке
pipe.fit(X_train, y_train)
print("Модель обучена.")

Мин. кол-во образцов в классе: 19 -> n_splits=5
CV accuracy: 0.9483 ± 0.0034
Модель обучена.


In [94]:
y_train_pred = pipe.predict(X_train)
print("Train accuracy:", f"{np.mean(y_train_pred == y_train):.4f}")
print("\nClassification report (train):")
print(classification_report(y_train, y_train_pred, digits=4))

Train accuracy: 1.0000

Classification report (train):
              precision    recall  f1-score   support

           0     1.0000    1.0000    1.0000      2335
           1     1.0000    1.0000    1.0000      1414
           2     1.0000    1.0000    1.0000        77
           3     1.0000    1.0000    1.0000       155
           4     1.0000    1.0000    1.0000        19

    accuracy                         1.0000      4000
   macro avg     1.0000    1.0000    1.0000      4000
weighted avg     1.0000    1.0000    1.0000      4000



In [95]:
y_pred = pipe.predict(X_test)

out = pd.DataFrame({
    "heartbeat_id": test_ids,
    "prediction":  y_pred.astype(int),
})

# --- Проверка контракта ---
assert len(out) == len(df_test), "Число строк не совпадает с test_features.csv"
assert out["heartbeat_id"].is_unique, "heartbeat_id должен быть уникальным"
assert out["prediction"].notna().all(), "Есть NaN в prediction"
assert set(out["prediction"].unique()).issubset(set(range(5))), \
       f"Классы должны быть 0..4, а получены {out['prediction'].unique()}"

out_path = "outputs/task2_1_predictions.csv"
out.to_csv(out_path, index=False)

print("Сохранено:", out_path)
print(out.head())
print("\nРаспределение предсказаний:")
print(out["prediction"].value_counts().sort_index())

Сохранено: outputs/task2_1_predictions.csv
   heartbeat_id  prediction
0             2           0
1             5           0
2            12           0
3            17           0
4            20           0

Распределение предсказаний:
prediction
0    591
1    382
2      8
3     19
Name: count, dtype: int64


In [96]:
check = pd.read_csv("outputs/task2_1_predictions.csv")
print("Файл:", "outputs/task2_1_predictions.csv")
print("Колонки:", list(check.columns))
print("Строк:", len(check))
print("Уникальных heartbeat_id:", check["heartbeat_id"].nunique())
print("Пропусков:", check.isna().sum().sum())
print("Классы:", sorted(check["prediction"].unique()))

assert list(check.columns) == ["heartbeat_id", "prediction"], "Колонки не те"
assert len(check) == len(df_test), "Число строк не совпадает"
assert check["heartbeat_id"].is_unique
assert check["prediction"].between(0, 4).all()
print("\n✅ Контракт Task 2.1 выполнен.")

Файл: outputs/task2_1_predictions.csv
Колонки: ['heartbeat_id', 'prediction']
Строк: 1000
Уникальных heartbeat_id: 1000
Пропусков: 0
Классы: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)]

✅ Контракт Task 2.1 выполнен.
